# CRMLS sold properties — Week 2 exploration

Load six consecutive monthly CSVs, verify their dates and schemas, restrict to residential single family homes, and inspect distributions. Raw data stays on your computer; do not commit CSVs or property level outputs to GitHub.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Set this to the folder containing your downloaded CRMLSSold CSVs.
DATA_DIR = Path.home() / 'Downloads' / 'idx exchange' / 'csv'
MONTHS = pd.period_range('2025-11', '2026-04', freq='M')
assert len(MONTHS) >= 6
print('Data folder:', DATA_DIR)


## 1. Locate and check each month
The filename can include `_filled`. We require exactly one matching CSV for each month and verify that its `CloseDate` belongs to the month. If April 2026 is incomplete or a newer complete month exists, change `MONTHS` above after checking with the team.


In [ ]:
required = {'CloseDate', 'ClosePrice', 'PropertyType', 'PropertySubType', 'LivingArea',
            'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet'}
frames = []
for month in MONTHS:
    matches = sorted(DATA_DIR.glob(f'CRMLSSold{month.strftime("%Y%m")}*.csv'))
    if len(matches) != 1:
        raise FileNotFoundError(f'{month}: expected exactly one CSV, found {matches}')
    df_month = pd.read_csv(matches[0], low_memory=False)
    missing = required - set(df_month.columns)
    if missing:
        raise ValueError(f'{matches[0].name}: missing columns {sorted(missing)}')
    dates = pd.to_datetime(df_month['CloseDate'], errors='coerce', utc=True)
    outside = dates.isna() | (dates.dt.strftime('%Y-%m') != str(month))
    if outside.any():
        raise ValueError(f'{matches[0].name}: {outside.sum()} invalid/out-of-month CloseDate values')
    df_month['source_month'] = str(month)
    frames.append(df_month)
    print(month, matches[0].name, f'{len(df_month):,} rows', f'{len(df_month.columns)-1} source columns')

raw = pd.concat(frames, ignore_index=True)
print(f'Total: {len(raw):,} rows; {len(raw.columns)-1} source columns')


## 2. Apply the task's property filters
Keep the downloaded source files unchanged. This filtered DataFrame is for analysis only.


In [ ]:
homes = raw.loc[(raw['PropertyType'] == 'Residential') &
                (raw['PropertySubType'] == 'SingleFamilyResidence')].copy()
features = ['ClosePrice', 'LivingArea', 'BedroomsTotal',
            'BathroomsTotalInteger', 'LotSizeSquareFeet']
for column in features:
    homes[column] = pd.to_numeric(homes[column], errors='coerce')
summary = homes.groupby('source_month').agg(
    homes=('ClosePrice', 'size'),
    median_price=('ClosePrice', 'median'),
    min_price=('ClosePrice', 'min'),
    max_price=('ClosePrice', 'max'),
)
display(summary)
print('Filtered rows:', f'{len(homes):,}')
print('Missing values by month:')
display(homes.groupby('source_month')[features].apply(lambda x: x.isna().sum()))
if 'ListingKey' in homes:
    print('Repeated non-null ListingKey values:', homes['ListingKey'].dropna().duplicated().sum())


## 3. Summaries and distributions
Inspect outliers before deciding how to handle them. The price plot is capped at the 99th percentile **for display only**; no rows are deleted. Confirm definitions and units against the Trestle metadata when available.


In [ ]:
display(homes[features].describe(percentiles=[.01, .25, .5, .75, .95, .99]).T)
display(homes[features].isna().mean().mul(100).round(2).rename('missing_pct'))
price_cap = homes['ClosePrice'].quantile(.99)
fig, ax = plt.subplots(figsize=(8, 4))
homes.loc[homes['ClosePrice'].between(0, price_cap), 'ClosePrice'].hist(bins=50, ax=ax)
ax.set(title='ClosePrice distribution (up to 99th percentile)', xlabel='Close price ($)', ylabel='Homes')
plt.tight_layout()
plt.show()


In [ ]:
plot_cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, column in zip(axes.flat, plot_cols):
    values = homes[column].dropna()
    values = values[values.between(0, values.quantile(.99))]
    values.hist(bins=40, ax=ax)
    ax.set(title=f'{column} (up to 99th percentile)', xlabel=column, ylabel='Homes')
plt.tight_layout()
plt.show()


In [ ]:
monthly = homes.groupby('source_month')['ClosePrice'].median()
ax = monthly.plot(marker='o', figsize=(8, 4), title='Median close price by month')
ax.set(xlabel='Close month', ylabel='Median close price ($)')
plt.tight_layout()
plt.show()
print('High-price records (no addresses or agent details):')
display(homes.nlargest(10, 'ClosePrice')[['source_month', 'ClosePrice', 'LivingArea',
                                           'BedroomsTotal', 'BathroomsTotalInteger',
                                           'LotSizeSquareFeet']])


## Findings to record
- Which six months were available, and did all `CloseDate` values match their filenames?
- How many single family residential records remained each month?
- Which key fields were missing, and how often?
- What were the typical ranges and notable outliers? Investigate before applying any cleaning rule.
- Do the distributions or row counts shift between months?

Week 3 will decide preprocessing and hold out the most recent complete month for testing. Do not choose cleaning rules based on test outcomes.
